# M66 R0 to A2 Vehicle feature distillation

Revision: `M66-R0-A2-VEHICLE-FEATURE-KD-2026-10-06-r1`.

Run these six sections in order on a GPU (A100 recommended). This is actual external-teacher distillation: frozen R0 ResNet50 teaches Vehicle depth-head features to A2 MobileNetV4 Conv Medium. Original A2 remains selected. MonoDGP/MonoPRIO and old M62 runtime recovery are not part of this notebook.

Two fresh arms each run exactly one epoch: native GT plus original-A2 preservation, and the same losses plus Vehicle feature KD. Only native depth MLPs update. Ground truth for both classes remains active; classification, boxes, dimensions, yaw, backbone, transformer and dense-depth predictor are frozen. Depth and uncertainty can still affect both classes and ranking, so full per-class evaluation is mandatory.

Requirements on Drive: original M65c manifest (used only for original A2 configuration/data identity), original A2 epoch130 weights, frozen R0 epoch185 selection/weights, Chen splits and KITTI data. No M65/M65c control checkpoint is a training parent. A passing M65c control is deliberately **not** required.

After a runtime restart rerun sections 1–4, then 5–6. Completed arms and prediction files are reused only with matching identity. A partial epoch restarts from original A2, not its last batch. GPU/software changes require a new RUN_ID. Never edit a historical manifest.

## 1 Setup paths and all helpers

This cell prints the revision and validates it against the script on main. Edit RUN_ID only for a deliberately fresh experiment.

In [ ]:
# M66-R0-A2-VEHICLE-FEATURE-KD-2026-10-06-r1 — self-contained paired pilot
from pathlib import Path
from collections import deque
import ast, json, os, runpy, shlex, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

REVISION = 'M66-R0-A2-VEHICLE-FEATURE-KD-2026-10-06-r1'
RUN_ID = 'm66_r0_a2_vehicle_feature_r1'
MOBILE_REPO = Path('/content/mobile_adas3d')
A2_REPO = Path('/content/MonoDETR_M66_A2')
A2_MANIFEST = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_m65c_scaled_preservation/m65c_a2_preservation_r1/m65c_manifest.json')
R0_SELECTION = Path('/content/drive/MyDrive/mobile_adas3d_outputs/references/monodetr_r0/product_checkpoint_sweep/r0_product_selection.json')
OUTPUT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_m66_r0_a2_feature')/RUN_ID
DATASET = Path('/content/kitti_m66')
SPLITS = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
PYTHON = Path('/content/m66_cuda130_venv/bin/python')
CUDA = Path('/usr/local/cuda-13.0')
if not (CUDA/'bin/nvcc').is_file(): CUDA = Path('/usr/local/cuda')
MANIFEST = OUTPUT/'m66_manifest.json'
SETUP_READY = MANIFEST_READY = TRAIN_READY = TRAIN_COMPLETE = False
ENV = os.environ.copy()
OUTPUT.mkdir(parents=True,exist_ok=True)
print(REVISION, '\nOutput:', OUTPUT, flush=True)

def run_logged(command, name, cwd=None):
    command = [str(x) for x in command]
    path = OUTPUT/'colab_logs'/f'{name}.log'
    path.parent.mkdir(parents=True,exist_ok=True)
    tail = deque(maxlen=60)
    print('+',shlex.join(command),'\nDurable log:',path,flush=True)
    with path.open('a') as log:
        log.write('\nCOMMAND: '+shlex.join(command)+'\n'); log.flush()
        with subprocess.Popen(command,cwd=str(cwd) if cwd else None,env=ENV,
                              stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
            status = process.wait()
        log.write(f'\nEXIT CODE: {status}\n'); log.flush()
    if status: raise RuntimeError(f'Exit {status}; full log={path}\n'+'\n'.join(tail))

def checkout(path,url,commit):
    if '[' in url or ']' in url or not url.startswith('https://github.com/'):
        raise ValueError('Use a plain public GitHub URL, not Markdown')
    if not path.exists():
        run_logged(['git','clone','--no-checkout',url,path],f'clone_{path.name}')
        run_logged(['git','checkout','--detach',commit],f'pin_{path.name}',path)
    if not (path/'.git').is_dir(): raise RuntimeError(f'Preserve {path}; choose a fresh checkout path')
    head = subprocess.check_output(['git','rev-parse','HEAD'],cwd=path,text=True).strip()
    origin = subprocess.check_output(['git','remote','get-url','origin'],cwd=path,text=True).strip()
    if head != commit or origin.rstrip('/') != url:
        raise RuntimeError(f'Unexpected checkout identity: {path}; do not reset it')

def workflow(stage,*extra):
    command = [PYTHON,'-X','faulthandler','-u',SCRIPT,stage]
    command += ['--output',OUTPUT] if stage in ('prepare','bundle') else ['--manifest',MANIFEST]
    name = 'm66_'+stage
    if '--model' in extra: name += '_'+str(extra[extra.index('--model')+1])
    run_logged(command+list(extra),name,MOBILE_REPO)

if not MOBILE_REPO.exists():
    run_logged(['git','clone','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone_mobile')
branch = subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch != 'main': raise RuntimeError(f'Use main, not {branch!r}; preserve other branch work')
run_logged(['git','pull','--ff-only'],'update_mobile',MOBILE_REPO)
SCRIPT = MOBILE_REPO/'scripts/m66_r0_a2_feature_pilot.py'
if not SCRIPT.is_file(): raise FileNotFoundError('M66 is not published yet; reload the latest main notebook.')
revisions = [ast.literal_eval(node.value) for node in ast.parse(SCRIPT.read_text()).body
             if isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='REVISION' for t in node.targets)]
if revisions != [REVISION]: raise RuntimeError('Notebook/script revisions differ; reload the latest notebook')


## 2 Build the private runtime and model source

Reuse the tested CUDA13 setup, lazy evaluator imports and GPU-specific attention builder. No old Colab package reconstruction or destructive git reset.

In [ ]:
SETUP_READY = MANIFEST_READY = TRAIN_READY = TRAIN_COMPLETE = False
if not SCRIPT.is_file(): raise RuntimeError('Run section 1 first')
helpers = runpy.run_path(str(MOBILE_REPO/'scripts/setup_m64_runtime.py'),run_name='shared_cuda13_helpers')
ENV = helpers['runtime_env'](CUDA,PYTHON.parent)
run_logged([sys.executable,'-u',MOBILE_REPO/'scripts/setup_m64_runtime.py',
            '--venv',PYTHON.parent.parent,'--cuda',CUDA,'--receipt',OUTPUT/'m66_runtime_receipt.json'],
           'm66_runtime',MOBILE_REPO)
checkout(A2_REPO,'https://github.com/ZrrSkywalker/MonoDETR.git','6994b9f512400b258c6edb75f77423beb9c126f2')
for patch in ('patch_monodetr_colab_compat.py','patch_monodetr_product_taxonomy.py','patch_monodetr_mobilenetv4.py'):
    run_logged([PYTHON,MOBILE_REPO/'scripts'/patch,'--monodetr-repo',A2_REPO],patch.removesuffix('.py'),MOBILE_REPO)
run_logged([PYTHON,MOBILE_REPO/'scripts/patch_m64_inference_sources.py','--repo',A2_REPO,'--role','a2'],'m66_patch_a2',MOBILE_REPO)
run_logged([PYTHON,'-X','faulthandler','-u',MOBILE_REPO/'scripts/build_m64_attention.py',
            '--repo',A2_REPO,'--role','a2'],'m66_build_a2',MOBILE_REPO)
for test in ('test_m64_teacher_qualification.py','test_m66_feature_pilot.py'):
    run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p',test,'-v'],test.removesuffix('.py'),MOBILE_REPO)
SETUP_READY = True
print('Section 2 passed. R0 and A2 use the same pinned MonoDETR source with different backbones.')


## 3 Restore data and freeze both models

Uses original A2, not the failed M65c checkpoint. A fresh prospective manifest authorizes only this bounded paired pilot; historical authorizations and hashes remain unchanged.

In [ ]:
MANIFEST_READY = TRAIN_READY = TRAIN_COMPLETE = False
if not SETUP_READY: raise RuntimeError('Run sections 1 and 2 first')
for path in (A2_MANIFEST,R0_SELECTION):
    if not path.is_file(): raise FileNotFoundError(path)
candidates = ['/content/kitti_m65c','/content/kitti_m65','/content/kitti_m64',
              '/content/kitti_m62','/content/kitti','/content/monodetr_kitti',
              '/content/monodetr_kitti_a2','/content/drive/MyDrive/datasets/kitti']
command = [PYTHON,MOBILE_REPO/'scripts/restore_m63_data.py','--dataset',DATASET,'--splits',SPLITS]
for candidate in candidates: command += ['--candidate',candidate]
run_logged(command,'m66_restore_data',MOBILE_REPO)
workflow('prepare','--a2-manifest',A2_MANIFEST,'--r0-selection',R0_SELECTION,
         '--a2-repo',A2_REPO,'--dataset-root',DATASET)
manifest = json.loads(MANIFEST.read_text())
assert manifest['kd_authorized'] is True and manifest['recipe']['epochs']==1
assert not manifest['passing_preservation_control_required'] and not manifest['monoprio_kd_authorized']
assert not manifest['full_training_authorized'] and not manifest['deployment_authorized']
MANIFEST_READY = True
print('Section 3 passed. External teacher: frozen R0 epoch185, Vehicle only. Student: original A2 epoch130.')


## 4 Fresh full baseline and real zero update KD smoke

Evaluate original A2 on all 3,769 validation images. Then verify actual teacher and student features, train-only masks, finite nonzero head gradients, and unchanged frozen state over 64 training images. This is not another preservation-only experiment.

In [ ]:
TRAIN_READY = TRAIN_COMPLETE = False
if not MANIFEST_READY: raise RuntimeError('Run sections 1–3 first')
workflow('infer','--model','baseline')
workflow('metrics','--model','baseline')
workflow('baseline')
workflow('smoke')
smoke = json.loads((OUTPUT/'m66_training_smoke.json').read_text())
assert smoke['optimizer_steps']==0 and smoke['passed'] and smoke['eligible_vehicle_pairs']>=8
assert all(smoke['checks'].values())
TRAIN_READY = True
print('Section 4 passed. Both paired one-epoch arms may now run from original A2.')


## 5 Run the matched control and actual distillation

Each arm starts anew from A2 epoch130, seed444, batch4, 928 updates, AdamW LR1e-6, no AMP. Only depth-head MLPs learn. Both get native GT plus 0.10 × scaled A2 preservation; KD additionally gets 0.10 × Vehicle cosine feature loss.

A failed *accuracy* preservation result in the control does not prevent running the matched KD arm. Failed setup, source identity, numerical smoke or incomplete training still stops execution.

In [ ]:
if not TRAIN_READY: raise RuntimeError('Run sections 1–4; do not bypass failed preflight')
TRAIN_COMPLETE = False
for role in ('control','kd'):
    workflow('train','--model',role)
    summary = json.loads((OUTPUT/role/'training_summary.json').read_text())
    assert summary['complete'] and summary['optimizer_steps']==928
    assert summary['external_kd_enabled']==(role=='kd')
    assert summary['frozen_parameters_unchanged'] and summary['teacher_unchanged'] and summary['anchor_unchanged']
TRAIN_COMPLETE = True
print('Both one-epoch arms complete. Run section 6; do not add epochs or change losses.')


## 6 Full evaluation and result bundle

Measure both arms on all 3,769 images. KD needs at least +0.15 Vehicle moderate 3D AP points versus **both** original A2 and the control; no more than 0.15-point loss in the other three moderate AP metrics or 0.005 nearby-recall loss for either class versus either comparator.

Report the original product gates separately. A promising pilot needs another seed before longer training. Stop for review regardless of pass. Return `m66_results.zip`; large checkpoint and prediction files remain on Drive.

In [ ]:
try:
    if TRAIN_COMPLETE:
        for role in ('control','kd'):
            workflow('infer','--model',role)
            workflow('metrics','--model',role)
        workflow('review')
        report = json.loads((OUTPUT/'m66_paired_gate.json').read_text())
        print('Pilot passed:',report['pilot_passed'])
        print('KD minus original A2:',report['kd_minus_original'])
        print('KD minus matched control:',report['kd_minus_control'])
    else:
        print('Partial diagnostics only; no completed pilot claim.')
finally:
    workflow('bundle')
    print('Return:',OUTPUT/'m66_results.zip')
print('STOP FOR REVIEW. Original A2 remains selected. No automatic promotion, extra epochs or phone deployment.')
